# Tải thư viện cần thiết

In [1]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "pyarrow", "pandas", "holidays", "pytz", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', 'holidays', 'pytz', '-q'], returncode=0)

# Khởi tạo SparkSession

In [2]:
import os
import sys
from pyspark.sql import SparkSession

ACCOUNT_NAME = os.environ["AZURE_STORAGE_ACCOUNT_NAME"]
ACCOUNT_KEY  = os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
CONTAINER    = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")
CONN_STR     = os.environ["AZURE_STORAGE_CONNECTION_STRING"]

spark = (SparkSession.builder
    .appName("Silver_to_Gold")
    .master("spark://spark-master:7077")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.pyspark.python", "/usr/bin/python3")
    .config("spark.executorEnv.PYSPARK_PYTHON", "/usr/bin/python3")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-azure:3.3.4,"
            "io.delta:delta-spark_2.13:4.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config(f"spark.hadoop.fs.azure.account.key.{ACCOUNT_NAME}.blob.core.windows.net", ACCOUNT_KEY)
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Spark version: 4.0.0


# Config path

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = f"wasbs://{CONTAINER}@{ACCOUNT_NAME}.blob.core.windows.net"
SILVER_PATH = f"{BASE_PATH}/silver/traffic_cleaned"
LOOKUP_PATH = f"{BASE_PATH}/artifacts/free_flow_speed_lookup.parquet"
GOLD_PATH   = f"{BASE_PATH}/gold/training_features"

print(f"BASE_PATH: {BASE_PATH}")

BASE_PATH: wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net


# Đọc Silver

In [4]:
df_silver = spark.read.format("delta").load(SILVER_PATH)

print("Tổng rows Silver:", df_silver.count())
print("Số link_id unique:", df_silver.select("link_id").distinct().count())

df_silver.printSchema()
df_silver.show(5, truncate=False)

Tổng rows Silver: 23522760
Số link_id unique: 116
root
 |-- link_id: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- link_name: string (nullable = true)
 |-- data_as_of: timestamp (nullable = true)
 |-- speed: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_holiday: integer (nullable = true)
 |-- year: integer (nullable = true)

+-------+-------------+-----------------------------------+-------------------+-----+----+-----------+-----+----------+----------+----+
|link_id|borough      |link_name                          |data_as_of         |speed|hour|day_of_week|month|is_weekend|is_holiday|year|
+-------+-------------+-----------------------------------+-------------------+-----+----+-----------+-----+----------+----------+----+
|4329499|Staten Island|MLK N WALKER STREET - NJ ROUTE 169 |2024-04-01 00:04:03|54.68|0   |2  

# tách Train set

In [5]:
# Train: 01/2024 -> 09/2025
df_train = df_silver.filter(
    (F.col("year") == 2024) |
    ((F.col("year") == 2025) & (F.col("month") <= 9))
)

print(f"Train rows: {df_train.count()}")
print(f"Train link_id unique: {df_train.select('link_id').distinct().count()}")

Train rows: 16914113
Train link_id unique: 116


# Tính free_flow_speed = P85(speed) theo link_id, chỉ trên Train

In [6]:
df_p85 = (df_train.groupBy("link_id")
    .agg(F.expr("percentile_approx(speed, 0.85)").alias("free_flow_speed"))
)

# borough, link_name là thuộc tính tĩnh của mỗi link -> lấy first() là đủ
df_link_meta = (df_train.groupBy("link_id")
    .agg(F.first("borough").alias("borough"),
         F.first("link_name").alias("link_name"))
)

df_lookup = df_p85.join(df_link_meta, "link_id", "left")

n_lookup = df_lookup.count()
print(f"Số link trong lookup: {n_lookup}")
df_lookup.orderBy("link_id").show(10)

Số link trong lookup: 116
+-------+---------------+-------------+--------------------+
|link_id|free_flow_speed|      borough|           link_name|
+-------+---------------+-------------+--------------------+
|4329472|           29.2|    Manhattan|LINCOLN TUNNEL E ...|
|4329473|          31.68|    Manhattan|LINCOLN TUNNEL E ...|
|4329499|          54.05|Staten Island|MLK N WALKER STRE...|
|4329507|          39.14|    Manhattan|LINCOLN TUNNEL W ...|
|4329508|          33.55|    Manhattan|LINCOLN TUNNEL W ...|
|4362244|           55.3|       Queens|CVE NB LIE - WILL...|
|4362249|          50.95|       Queens|LIE WB LITTLE NEC...|
|4362250|          60.27|       Queens|CVE NB GCP - WILL...|
|4362251|          53.43|       Queens|LIE WB LITTLE NEC...|
|4362314|          50.95|       Queens|TNB S Qns Anchora...|
+-------+---------------+-------------+--------------------+
only showing top 10 rows


In [7]:
df_lookup.write.mode("overwrite").parquet(LOOKUP_PATH)
print(f"Ghi free_flow_speed_lookup.parquet → {LOOKUP_PATH} ({n_lookup} rows)")

Ghi free_flow_speed_lookup.parquet → wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/artifacts/free_flow_speed_lookup.parquet (116 rows)


# Join lookup vào toàn bộ Silver

In [8]:
# Silver đã có borough, link_name sẵn -> chỉ cần lấy free_flow_speed từ lookup
df_gold = df_silver.join(
    df_lookup.select("link_id", "free_flow_speed"),
    on="link_id",
    how="left"
)

print(f"Rows sau join: {df_gold.count()}")
print(f"Rows free_flow_speed null (link thiếu lookup): "
      f"{df_gold.filter(F.col('free_flow_speed').isNull()).count()}")

Rows sau join: 23522760
Rows free_flow_speed null (link thiếu lookup): 0


# speed_ratio và current_congestion

In [9]:
df_gold = df_gold.withColumn(
    "speed_ratio",
    F.col("speed") / F.col("free_flow_speed")
)

# Ngưỡng 0.67 / 0.40 -> 3 lớp:
#   0 = thông thoáng   (speed_ratio >= 0.67)
#   1 = tắc trung bình (0.40 <= speed_ratio < 0.67)
#   2 = tắc nặng        (speed_ratio < 0.40)
# QUAN TRỌNG: phải check isNull() trước, nếu không Spark sẽ rơi vào .otherwise(2) sai cho các giá trị null
df_gold = df_gold.withColumn(
    "current_congestion",
    F.when(F.col("speed_ratio").isNull(), F.lit(None))
     .when(F.col("speed_ratio") >= 0.67, F.lit(0))
     .when(F.col("speed_ratio") >= 0.40, F.lit(1))
     .otherwise(F.lit(2))
     .cast("int")
)

df_gold.groupBy("current_congestion").count().orderBy("current_congestion").show()

+------------------+--------+
|current_congestion|   count|
+------------------+--------+
|                 0|15915889|
|                 1| 3465650|
|                 2| 4141221|
+------------------+--------+



# Lag 15/30/45/60 phút + Lead 15 phút

In [10]:
window_link = Window.partitionBy("link_id").orderBy("data_as_of")

lag_map = {15: 3, 30: 6, 45: 9, 60: 12}  # phút : số dòng lag (giả định ~5 phút/dòng)

for minutes, offset in lag_map.items():
    df_gold = df_gold.withColumn(
        f"past_congestion_{minutes}min",
        F.lag("current_congestion", offset).over(window_link)
    )
    df_gold = df_gold.withColumn(
        f"past_speed_ratio_{minutes}min",
        F.lag("speed_ratio", offset).over(window_link)
    )
    # cột phụ để validate gap thời gian thực tế, sẽ drop sau
    df_gold = df_gold.withColumn(
        f"_past_ts_{minutes}min",
        F.lag("data_as_of", offset).over(window_link)
    )

# Label: future_congestion_15min = lead 3 dòng (~15 phút sau)
df_gold = df_gold.withColumn(
    "future_congestion_15min",
    F.lead("current_congestion", 3).over(window_link)
)
df_gold = df_gold.withColumn(
    "_future_ts_15min",
    F.lead("data_as_of", 3).over(window_link)
)

#  Validate gap thực tế, drop dòng lệch

In [11]:
tolerance = {15: (10, 20), 30: (25, 35), 45: (40, 50), 60: (55, 65)}

# Tính gap (phút) cho từng lag
for minutes in [15, 30, 45, 60]:
    df_gold = df_gold.withColumn(
        f"_gap_{minutes}min",
        (F.col("data_as_of").cast("long") - F.col(f"_past_ts_{minutes}min").cast("long")) / 60
    )

# Gap cho lead (future)
df_gold = df_gold.withColumn(
    "_gap_future_15min",
    (F.col("_future_ts_15min").cast("long") - F.col("data_as_of").cast("long")) / 60
)

# Điều kiện hợp lệ: nếu lag/lead null (do ở biên link) -> tạm cho qua, sẽ bị loại ở bước Drop NaN sau.
# Nếu lag/lead có giá trị -> gap phải nằm trong khung dung sai.
def valid_gap(col_gap, lo, hi):
    return F.col(col_gap).isNull() | ((F.col(col_gap) >= lo) & (F.col(col_gap) <= hi))

cond = valid_gap("_gap_future_15min", 10, 20)
for minutes, (lo, hi) in tolerance.items():
    cond = cond & valid_gap(f"_gap_{minutes}min", lo, hi)

rows_before = df_gold.count()
df_gold = df_gold.filter(cond)
rows_after = df_gold.count()
print(f"Trước validate gap: {rows_before:,} | Sau: {rows_after:,} | Đã loại: {rows_before - rows_after:,}")

Trước validate gap: 23,522,760 | Sau: 21,888,843 | Đã loại: 1,633,917


# Drop NaN (bất kỳ lag/lead nào null)

In [12]:
required_for_dropna = (
    [f"past_congestion_{m}min" for m in [15, 30, 45, 60]] +
    [f"past_speed_ratio_{m}min" for m in [15, 30, 45, 60]] +
    ["future_congestion_15min", "speed_ratio", "current_congestion"]
)

rows_before = df_gold.count()
df_gold = df_gold.dropna(subset=required_for_dropna)
rows_after = df_gold.count()
print(f"Trước dropna: {rows_before:,} | Sau: {rows_after:,} | Đã loại: {rows_before - rows_after:,}")

Trước dropna: 21,888,843 | Sau: 21,887,219 | Đã loại: 1,624


# Tính 2 cột trend

In [13]:
df_gold = df_gold.withColumn(
    "congestion_trend",
    F.col("current_congestion") - F.col("past_congestion_15min")
)
df_gold = df_gold.withColumn(
    "speed_ratio_trend",
    F.col("speed_ratio") - F.col("past_speed_ratio_15min")
)

# Chọn đúng schema Gold (19 features + label + data_as_of) và ghi Delta

In [14]:
feature_cols = [
    "link_id", "borough", "hour", "day_of_week", "month", "is_weekend", "is_holiday",   # 7
    "speed_ratio", "current_congestion",                                                # 9
    "past_congestion_15min", "past_speed_ratio_15min",                                  # 11
    "past_congestion_30min", "past_speed_ratio_30min",                                  # 13
    "past_congestion_45min", "past_speed_ratio_45min",                                  # 15
    "past_congestion_60min", "past_speed_ratio_60min",                                  # 17
    "congestion_trend", "speed_ratio_trend",                                            # 19
]
assert len(feature_cols) == 19, f"Đếm lại! Hiện có {len(feature_cols)} cột"

df_gold_final = df_gold.select(
    *feature_cols,
    "future_congestion_15min",   # label
    "data_as_of",                 # mốc thời gian, không phải feature
    "year"                         # giữ để partition (month đã có trong feature_cols)
)

df_gold_final.printSchema()
print(f"Tổng cột (19 feature + 1 label + data_as_of + year): {len(df_gold_final.columns)}")

root
 |-- link_id: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_holiday: integer (nullable = true)
 |-- speed_ratio: double (nullable = true)
 |-- current_congestion: integer (nullable = true)
 |-- past_congestion_15min: integer (nullable = true)
 |-- past_speed_ratio_15min: double (nullable = true)
 |-- past_congestion_30min: integer (nullable = true)
 |-- past_speed_ratio_30min: double (nullable = true)
 |-- past_congestion_45min: integer (nullable = true)
 |-- past_speed_ratio_45min: double (nullable = true)
 |-- past_congestion_60min: integer (nullable = true)
 |-- past_speed_ratio_60min: double (nullable = true)
 |-- congestion_trend: integer (nullable = true)
 |-- speed_ratio_trend: double (nullable = true)
 |-- future_congestion_15min: integer (nullable = true)
 |-- data_as_of: timestamp (null

# Ghi Gold Delta, partition year, month

In [15]:
import time

# 1. Cache trước khi ghi — vì ta sẽ dùng df_gold_final 2 lần (đếm + ghi),
#    cache giúp lần ghi không phải tính lại toàn bộ window function/join từ đầu
df_gold_final = df_gold_final.repartition(64, "year", "month").cache()

# 2. Materialize cache bằng action đầu tiên + đo thời gian
t0 = time.time()
total_rows = df_gold_final.count()
print(f"[1/4] Đã tính xong toàn bộ pipeline: {total_rows:,} rows ({time.time()-t0:.1f}s)")

# 3. Log breakdown theo year/month TRƯỚC khi ghi — để biết khối lượng từng partition,
#    phát hiện sớm partition nào quá to (dễ OOM) trước khi bắt đầu ghi
print("[2/4] Phân bố rows theo year/month sẽ ghi:")
(df_gold_final.groupBy("year", "month")
    .count()
    .orderBy("year", "month")
    .show(40, truncate=False))

# 4. Ghi Delta + đo thời gian
print(f"[3/4] Bắt đầu ghi Gold Delta → {GOLD_PATH} ...")
t1 = time.time()

(df_gold_final.write.format("delta").mode("overwrite")
    .partitionBy("year", "month")
    .option("maxRecordsPerFile", 2_000_000)
    .save(GOLD_PATH))

print(f"Ghi xong sau {time.time()-t1:.1f}s")

# Giải phóng cache, không cần giữ trong memory nữa
df_gold_final.unpersist()

[1/4] Đã tính xong toàn bộ pipeline: 21,887,219 rows (129.5s)
[2/4] Phân bố rows theo year/month sẽ ghi:
+----+-----+------+
|year|month|count |
+----+-----+------+
|2023|12   |4512  |
|2024|1    |766767|
|2024|2    |462213|
|2024|3    |791258|
|2024|4    |782066|
|2024|5    |776141|
|2024|6    |782608|
|2024|7    |776870|
|2024|8    |741714|
|2024|9    |800545|
|2024|10   |817888|
|2024|11   |794750|
|2024|12   |666384|
|2025|1    |824472|
|2025|2    |691435|
|2025|3    |776612|
|2025|4    |758096|
|2025|5    |747874|
|2025|6    |688638|
|2025|7    |761101|
|2025|8    |816226|
|2025|9    |767461|
|2025|10   |779963|
|2025|11   |728302|
|2025|12   |762818|
|2026|1    |815809|
|2026|2    |688147|
|2026|3    |640131|
|2026|4    |496033|
|2026|5    |690488|
|2026|6    |489897|
+----+-----+------+

[3/4] Bắt đầu ghi Gold Delta → wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/gold/training_features ...
Ghi xong sau 204.4s


DataFrame[link_id: string, borough: string, hour: int, day_of_week: int, month: int, is_weekend: int, is_holiday: int, speed_ratio: double, current_congestion: int, past_congestion_15min: int, past_speed_ratio_15min: double, past_congestion_30min: int, past_speed_ratio_30min: double, past_congestion_45min: int, past_speed_ratio_45min: double, past_congestion_60min: int, past_speed_ratio_60min: double, congestion_trend: int, speed_ratio_trend: double, future_congestion_15min: int, data_as_of: timestamp, year: int]

# Verify

In [16]:
df_check = spark.read.format("delta").load(GOLD_PATH)
print(f"Tổng rows: {df_check.count():,}")
print(f"Link unique: {df_check.select('link_id').distinct().count()}")
df_check.groupBy("year", "month").count().orderBy("year", "month").show(40)
df_check.groupBy("future_congestion_15min").count().orderBy("future_congestion_15min").show()

Tổng rows: 21,887,219
Link unique: 116
+----+-----+------+
|year|month| count|
+----+-----+------+
|2023|   12|  4512|
|2024|    1|766767|
|2024|    2|462213|
|2024|    3|791258|
|2024|    4|782066|
|2024|    5|776141|
|2024|    6|782608|
|2024|    7|776870|
|2024|    8|741714|
|2024|    9|800545|
|2024|   10|817888|
|2024|   11|794750|
|2024|   12|666384|
|2025|    1|824472|
|2025|    2|691435|
|2025|    3|776612|
|2025|    4|758096|
|2025|    5|747874|
|2025|    6|688638|
|2025|    7|761101|
|2025|    8|816226|
|2025|    9|767461|
|2025|   10|779963|
|2025|   11|728302|
|2025|   12|762818|
|2026|    1|815809|
|2026|    2|688147|
|2026|    3|640131|
|2026|    4|496033|
|2026|    5|690488|
|2026|    6|489897|
+----+-----+------+

+-----------------------+--------+
|future_congestion_15min|   count|
+-----------------------+--------+
|                      0|15000112|
|                      1| 3227613|
|                      2| 3659494|
+-----------------------+--------+



In [18]:
df_check.printSchema()
df_check.groupBy("year", "month").count().orderBy("year", "month").show(50)
df_check.groupBy("future_congestion_15min").count().orderBy("future_congestion_15min").show()

root
 |-- link_id: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_holiday: integer (nullable = true)
 |-- speed_ratio: double (nullable = true)
 |-- current_congestion: integer (nullable = true)
 |-- past_congestion_15min: integer (nullable = true)
 |-- past_speed_ratio_15min: double (nullable = true)
 |-- past_congestion_30min: integer (nullable = true)
 |-- past_speed_ratio_30min: double (nullable = true)
 |-- past_congestion_45min: integer (nullable = true)
 |-- past_speed_ratio_45min: double (nullable = true)
 |-- past_congestion_60min: integer (nullable = true)
 |-- past_speed_ratio_60min: double (nullable = true)
 |-- congestion_trend: integer (nullable = true)
 |-- speed_ratio_trend: double (nullable = true)
 |-- future_congestion_15min: integer (nullable = true)
 |-- data_as_of: timestamp (null